<a href="https://colab.research.google.com/github/ArsyadAptaa/Tugas-KKA-Praktikum-EDA/blob/main/Praktik_Modul_EDA_Kelompok.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Project Charter - Proyek EDA Kelompok

**Nama Anggota Kelompok:**
1. Achamd Ryu Mariadi (1)
2. Muhammad Arsyad Apta W (20)

**Dataset yang Dipilih:**
Data Penjualan Kantin/Toko

**Pertanyaan Analisis Awal:**
1. Produk apa yang paling laris (jumlah terjual terbanyak)?
2. Produk apa yang menyumbang total pendapatan tertinggi?
3. Metode pembayaran apa yang paling sering dipakai pelanggan kantin?

**Dugaan Masalah Kualitas Data:**
- Missing value pada kolom jumlah_terjual, harga_satuan, dan nama_kasir
- Ada baris data yang duplikat
- Tipe data tidak sesuai: harga_satuan bercampur teks "Rp", jumlah_terjual ada satuan "pcs",
  format tanggal tidak seragam
- Penulisan kategori tidak konsisten (huruf besar/kecil campur), dan ada outlier pada jumlah_terjual

**Rencana Teknik Pembersihan:**
Menyeragamkan format tanggal, membersihkan kolom harga_satuan dan jumlah_terjual dari teks
tambahan lalu mengubahnya ke angka, mengisi data kosong dengan nilai yang masuk akal
(harga umum produk, angka 0, atau keterangan "Tidak Diketahui"), menyeragamkan penulisan
kategori, serta menghapus baris duplikat.

**Rencana Manipulasi Data:**
- Filter: transaksi dengan jumlah_terjual lebih dari 10
- Sort: berdasarkan total_pendapatan dari yang tertinggi
- Kolom turunan: total_pendapatan = harga_satuan x jumlah_terjual
- Groupby/agregasi: total_pendapatan per nama_produk dan per kategori

In [ ]:
import pandas as pd
import numpy as np
from google.colab import files

uploaded = files.upload()

## pandas dipakai untuk mengolah data tabel, numpy untuk operasi angka.
## Baris files.upload(), dipakai supaya kita bisa mengunggah file CSV langsung dari komputer ke notebook.

Saving dataset_penjualan_kantin.csv to dataset_penjualan_kantin (1).csv


In [ ]:
df = pd.read_csv('dataset_penjualan_kantin.csv')

print("5 baris pertama:")
print(df.head())

print("\nInfo dataset:")
print(df.info())

print("\nStatistik ringkas:")
print(df.describe())

print("\nJumlah baris dan kolom:", df.shape)

## pd.read_csv() digunakan untuk membuka file CSV menjadi tabel.
## head() menampilkan 5 data pertama. info() melihat jenis data dan data yang kosong.
## describe() menampilkan statistik seperti rata-rata, nilai terkecil, dan terbesar.
## shape menunjukkan jumlah baris dan kolom.

5 baris pertama:
  id_transaksi     tanggal  nama_produk kategori jumlah_terjual harga_satuan  \
0      TRX0042  2026-08-11   Roti Bakar  Makanan              9         7000   
1      TRX0005  2026-08-03     Gorengan  makanan              2         2000   
2      TRX0011  2026-08-04  Jus Alpukat  Minuman              4          NaN   
3      TRX0035  2026-08-10     Mie Ayam  Makanan            NaN        10000   
4      TRX0007  2026-08-03      Kerupuk    Snack             10      Rp2.000   

  nama_kasir metode_pembayaran  
0   Pak Agus             Tunai  
1     Bu Sri              QRIS  
2    Bu Wati             Tunai  
3   Pak Joko          Transfer  
4    Bu Wati          Transfer  

Info dataset:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 69 entries, 0 to 68
Data columns (total 8 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   id_transaksi       69 non-null     object
 1   tanggal            69 non-null     object

In [ ]:
print("Jumlah data kosong tiap kolom:")
print(df.isnull().sum())

print("\nJumlah baris duplikat:", df.duplicated().sum())

## isnull().sum() menghitung jumlah data yang kosong di setiap kolom.
## duplicated().sum() menghitung jumlah data yang sama/duplikat.
## Langkah ini dilakukan untuk mengetahui masalah dalam data sebelum data dibersihkan.

Jumlah data kosong tiap kolom:
id_transaksi         0
tanggal              0
nama_produk          0
kategori             0
jumlah_terjual       4
harga_satuan         3
nama_kasir           3
metode_pembayaran    0
dtype: int64

Jumlah baris duplikat: 4


In [ ]:
def bersihkan_tanggal(x):
    x = str(x).strip()
    for fmt in ('%Y-%m-%d', '%d/%m/%Y', '%d %B %Y'):
        try:
            return pd.to_datetime(x, format=fmt)
        except ValueError:
            continue
    return pd.NaT

bulan_indonesia = {
    'Januari': 'January', 'Februari': 'February', 'Maret': 'March', 'April': 'April',
    'Mei': 'May', 'Juni': 'June', 'Juli': 'July', 'Agustus': 'August',
    'September': 'September', 'Oktober': 'October', 'November': 'November', 'Desember': 'December'
}
for id_bulan, en_bulan in bulan_indonesia.items():
    df['tanggal'] = df['tanggal'].astype(str).str.replace(id_bulan, en_bulan, regex=False)

df['tanggal'] = df['tanggal'].apply(bersihkan_tanggal)
print(df['tanggal'].head(10))

## Kolom tanggal memiliki format yang berbeda-beda. Kode ini menyamakan semua format tanggal agar bisa diurutkan dan dianalisis.

0   2026-08-11
1   2026-08-03
2   2026-08-04
3   2026-08-10
4   2026-08-03
5   2026-08-11
6   2026-08-14
7   2026-08-04
8   2026-08-07
9   2026-08-13
Name: tanggal, dtype: datetime64[ns]


In [ ]:
df['kategori'] = df['kategori'].str.strip().str.title()
print(df['kategori'].unique())

## Kolom kategori memiliki penulisan yang berbeda seperti MAKANAN, makanan, dan Makanan.
## Kode ini menyamakan penulisannya dan menghapus spasi berlebih.

['Makanan' 'Minuman' 'Snack']


In [ ]:
df['harga_satuan'] = (
    df['harga_satuan']
    .astype(str)
    .str.replace('Rp', '', regex=False)
    .str.replace('.', '', regex=False)
    .str.strip()
)
df['harga_satuan'] = pd.to_numeric(df['harga_satuan'], errors='coerce')

# isi harga yang kosong dengan harga umum (modus) dari produk yang sama
df['harga_satuan'] = df.groupby('nama_produk')['harga_satuan'].transform(
    lambda x: x.fillna(x.mode()[0] if not x.mode().empty else x.mean())
)

df['harga_satuan'] = df['harga_satuan'].astype(int)
print(df['harga_satuan'].head(10))

## Kolom harga_satuan berisi angka dan tulisan seperti "Rp7.000".
## Kode ini menghapus "Rp" dan titik, lalu mengubahnya menjadi angka.
## Harga yang kosong diisi berdasarkan harga yang paling sering muncul pada produk yang sama.

0     7000
1     2000
2     8000
3    10000
4     2000
5     5000
6     5000
7    10000
8    12000
9     8000
Name: harga_satuan, dtype: int64


In [ ]:
df['jumlah_terjual'] = (
    df['jumlah_terjual']
    .astype(str)
    .str.replace('pcs', '', regex=False)
    .str.strip()
)
df['jumlah_terjual'] = pd.to_numeric(df['jumlah_terjual'], errors='coerce')

# isi data kosong dengan 0 (dianggap tidak ada transaksi tercatat)
df['jumlah_terjual'] = df['jumlah_terjual'].fillna(0)

# tangani outlier: nilai yang jauh di luar kewajaran (misal > 100) dianggap salah input
df.loc[df['jumlah_terjual'] > 100, 'jumlah_terjual'] = df['jumlah_terjual'].median()

df['jumlah_terjual'] = df['jumlah_terjual'].astype(int)
print(df['jumlah_terjual'].describe())

## Kolom jumlah_terjual memiliki tulisan "pcs" sehingga harus dihapus agar menjadi angka.
## Data kosong diisi 0, sedangkan nilai 500 yang tidak wajar diganti dengan nilai median.

count    69.000000
mean      6.985507
std       4.577736
min       0.000000
25%       3.000000
50%       7.000000
75%      10.000000
max      15.000000
Name: jumlah_terjual, dtype: float64


In [ ]:
df['nama_kasir'] = df['nama_kasir'].fillna('Tidak Diketahui')

print("Jumlah baris sebelum hapus duplikat:", len(df))
df = df.drop_duplicates()
print("Jumlah baris sesudah hapus duplikat:", len(df))

print("\nCek ulang missing value:")
print(df.isnull().sum())
print("\nCek ulang tipe data:")
print(df.dtypes)

## Nama kasir yang kosong diisi dengan "Tidak Diketahui". drop_duplicates() digunakan untuk menghapus data yang sama.
## Setelah itu, data dicek kembali untuk memastikan sudah bersih.

Jumlah baris sebelum hapus duplikat: 69
Jumlah baris sesudah hapus duplikat: 65

Cek ulang missing value:
id_transaksi         0
tanggal              0
nama_produk          0
kategori             0
jumlah_terjual       0
harga_satuan         0
nama_kasir           0
metode_pembayaran    0
dtype: int64

Cek ulang tipe data:
id_transaksi                 object
tanggal              datetime64[ns]
nama_produk                  object
kategori                     object
jumlah_terjual                int64
harga_satuan                  int64
nama_kasir                   object
metode_pembayaran            object
dtype: object


In [ ]:
# kolom turunan: total pendapatan tiap transaksi
df['total_pendapatan'] = df['harga_satuan'] * df['jumlah_terjual']

# filtering: transaksi dengan penjualan tinggi (laris)
laris = df[df['jumlah_terjual'] > 10]
print("Transaksi dengan penjualan > 10 pcs:")
print(laris[['nama_produk', 'jumlah_terjual', 'total_pendapatan']].head())

# sorting: urutkan berdasarkan total_pendapatan tertinggi
urut_pendapatan = df.sort_values(by='total_pendapatan', ascending=False)
print("\n5 transaksi dengan pendapatan tertinggi:")
print(urut_pendapatan[['nama_produk', 'total_pendapatan']].head())

## total_pendapatan dibuat dengan mengalikan harga dan jumlah terjual.
## Data dengan penjualan lebih dari 10 disaring, lalu diurutkan dari pendapatan terbesar ke terkecil.

Transaksi dengan penjualan > 10 pcs:
         nama_produk  jumlah_terjual  total_pendapatan
6          Teh Botol              14             70000
7           Mie Ayam              14            140000
9        Jus Alpukat              14            112000
12         Teh Botol              12             60000
14  Keripik Singkong              15             45000

5 transaksi dengan pendapatan tertinggi:
    nama_produk  total_pendapatan
20  Nasi Goreng            180000
15  Nasi Goreng            156000
32  Nasi Goreng            144000
7      Mie Ayam            140000
44  Nasi Goreng            132000


In [ ]:
# agregasi 1: total pendapatan per produk
pendapatan_per_produk = df.groupby('nama_produk')['total_pendapatan'].sum().sort_values(ascending=False)
print("Total pendapatan per produk:")
print(pendapatan_per_produk)

# agregasi 2: total pendapatan per kategori
pendapatan_per_kategori = df.groupby('kategori')['total_pendapatan'].sum().sort_values(ascending=False)
print("\nTotal pendapatan per kategori:")
print(pendapatan_per_kategori)

# agregasi 3: metode pembayaran paling sering dipakai
metode_favorit = df['metode_pembayaran'].value_counts()
print("\nJumlah transaksi per metode pembayaran:")
print(metode_favorit)

## groupby() digunakan untuk menghitung total pendapatan berdasarkan produk dan kategori.
## value_counts() digunakan untuk mengetahui metode pembayaran yang paling sering digunakan.

Total pendapatan per produk:
nama_produk
Nasi Goreng         876000
Mie Ayam            510000
Jus Alpukat         496000
Bakso               352000
Teh Botol           290000
Roti Bakar          210000
Nasi Uduk           168000
Es Teh               93000
Kerupuk              92000
Keripik Singkong     72000
Es Jeruk             36000
Gorengan             18000
Name: total_pendapatan, dtype: int64

Total pendapatan per kategori:
kategori
Makanan    2134000
Minuman     915000
Snack       164000
Name: total_pendapatan, dtype: int64

Jumlah transaksi per metode pembayaran:
metode_pembayaran
QRIS        35
Tunai       23
Transfer     7
Name: count, dtype: int64


In [ ]:
df.to_csv('dataset_kantin_bersih.csv', index=False)
files.download('dataset_kantin_bersih.csv')
print("Dataset bersih berhasil disimpan dan siap dianalisis lebih lanjut.")

## to_csv() digunakan untuk menyimpan data yang sudah dibersihkan ke file CSV baru.
## files.download() digunakan untuk mengunduh file tersebut dari Google Colab.

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Dataset bersih berhasil disimpan dan siap dianalisis lebih lanjut.


Ringkasan Temuan Awal:

1. Setelah dibersihkan, data penjualan kantin menunjukkan bahwa Nasi Goreng dan Mie Ayam
   adalah dua produk dengan total pendapatan tertinggi, menandakan makanan berat lebih
   diminati dibanding minuman atau snack.

2. Metode pembayaran QRIS paling banyak digunakan pelanggan dibanding Tunai dan Transfer,
   menunjukkan kebiasaan pembayaran non-tunai sudah cukup dominan di kantin sekolah.

3. Dataset awal memiliki cukup banyak masalah kualitas data (format tanggal campur,
   penulisan kategori tidak konsisten, harga bercampur teks, dan beberapa data kosong/duplikat),
   yang menegaskan pentingnya tahap Data Cleaning sebelum data bisa dipakai untuk
   pengambilan keputusan yang akurat.